In [1]:
import duckdb

DATA_DIR = r"C:\Users\shyam\Downloads\synerise_dataset"

con = duckdb.connect("ecommerce_analytics.duckdb")

print("=" * 80)
print("BUILDING PRODUCT BEHAVIOR TABLE")
print("=" * 80)

query = f"""
CREATE OR REPLACE TABLE product_behavior AS

WITH

products AS (
    SELECT
        sku,
        category,
        price,
        name
    FROM read_parquet('{DATA_DIR}/product_properties.parquet')
),

cart_add AS (
    SELECT
        sku,
        COUNT(*) AS cart_additions,
        COUNT(DISTINCT client_id) AS unique_cart_customers
    FROM read_parquet('{DATA_DIR}/add_to_cart.parquet')
    GROUP BY sku
),

cart_remove AS (
    SELECT
        sku,
        COUNT(*) AS cart_removals,
        COUNT(DISTINCT client_id) AS unique_remove_customers
    FROM read_parquet('{DATA_DIR}/remove_from_cart.parquet')
    GROUP BY sku
),

purchases AS (
    SELECT
        sku,
        COUNT(*) AS purchase_events,
        COUNT(DISTINCT client_id) AS unique_purchase_customers
    FROM read_parquet('{DATA_DIR}/product_buy.parquet')
    GROUP BY sku
)

SELECT

    p.sku,
    p.category,
    p.price,
    p.name,

    COALESCE(ca.cart_additions, 0) AS cart_additions,
    COALESCE(ca.unique_cart_customers, 0) AS unique_cart_customers,

    COALESCE(cr.cart_removals, 0) AS cart_removals,
    COALESCE(cr.unique_remove_customers, 0) AS unique_remove_customers,

    COALESCE(pr.purchase_events, 0) AS purchase_events,
    COALESCE(pr.unique_purchase_customers, 0) AS unique_purchase_customers,

    CASE
        WHEN COALESCE(ca.cart_additions, 0) > 0
        THEN ROUND(
            COALESCE(pr.purchase_events, 0) * 1.0
            / ca.cart_additions,
            4
        )
        ELSE 0
    END AS purchase_per_cart_rate,

    CASE
        WHEN COALESCE(ca.cart_additions, 0) > 0
        THEN ROUND(
            COALESCE(cr.cart_removals, 0) * 1.0
            / ca.cart_additions,
            4
        )
        ELSE 0
    END AS cart_removal_rate

FROM products p

LEFT JOIN cart_add ca
    ON p.sku = ca.sku

LEFT JOIN cart_remove cr
    ON p.sku = cr.sku

LEFT JOIN purchases pr
    ON p.sku = pr.sku
"""

con.execute(query)

print("\nProduct behavior table created successfully!")


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("PRODUCT TABLE SUMMARY")
print("=" * 80)

result = con.execute("""
SELECT
    COUNT(*) AS products,
    COUNT(*) FILTER (WHERE purchase_events > 0) AS purchased_products,
    COUNT(*) FILTER (WHERE cart_additions > 0) AS carted_products,
    COUNT(*) FILTER (WHERE cart_removals > 0) AS removed_products
FROM product_behavior
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# TOP PRODUCTS BY PURCHASE EVENTS
# ============================================================

print("\n" + "=" * 80)
print("TOP PRODUCTS BY PURCHASE EVENTS")
print("=" * 80)

result = con.execute("""
SELECT
    sku,
    category,
    price,
    purchase_events,
    unique_purchase_customers,
    cart_additions,
    cart_removals
FROM product_behavior
WHERE purchase_events > 0
ORDER BY purchase_events DESC
LIMIT 20
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# TOP PRODUCTS BY UNIQUE CUSTOMERS
# ============================================================

print("\n" + "=" * 80)
print("TOP PRODUCTS BY UNIQUE PURCHASERS")
print("=" * 80)

result = con.execute("""
SELECT
    sku,
    category,
    price,
    unique_purchase_customers,
    purchase_events,
    cart_additions
FROM product_behavior
WHERE purchase_events > 0
ORDER BY unique_purchase_customers DESC
LIMIT 20
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# HIGH CART / LOW PURCHASE PRODUCTS
# ============================================================

print("\n" + "=" * 80)
print("HIGH CART ACTIVITY / LOW PURCHASE PRODUCTS")
print("=" * 80)

result = con.execute("""
SELECT
    sku,
    category,
    price,
    cart_additions,
    purchase_events,
    unique_cart_customers,
    unique_purchase_customers,
    purchase_per_cart_rate
FROM product_behavior
WHERE cart_additions >= 20
ORDER BY purchase_per_cart_rate ASC
LIMIT 20
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# CATEGORY SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("CATEGORY PERFORMANCE")
print("=" * 80)

result = con.execute("""
SELECT
    category,

    COUNT(*) AS products,

    SUM(cart_additions) AS cart_additions,

    SUM(cart_removals) AS cart_removals,

    SUM(purchase_events) AS purchase_events,

    COUNT(DISTINCT
        CASE
            WHEN purchase_events > 0 THEN sku
        END
    ) AS purchased_products

FROM product_behavior

GROUP BY category

ORDER BY purchase_events DESC

LIMIT 20
""").fetchdf()

print(result.to_string(index=False))


con.close()

print("\nDONE!")

BUILDING PRODUCT BEHAVIOR TABLE

Product behavior table created successfully!

PRODUCT TABLE SUMMARY
 products  purchased_products  carted_products  removed_products
  1534050              639126          1444160            824536

TOP PRODUCTS BY PURCHASE EVENTS
    sku  category  price  purchase_events  unique_purchase_customers  cart_additions  cart_removals
 219232       419     11             1957                       1510            9130           3313
  52586      3398     27             1563                       1212            7531           3163
 147718      5830     60             1461                       1005            2403            667
1243077      5934      3             1319                       1042            7374           2536
 940412      6794     47             1312                        955            3452           1217
 561077      1135     33             1294                        989            5677           1998
 342897      3207     69            